## Tracr Program

#### I'm writing 12 tracr programs and I'm trying to answer its residual lane and schedule as we go along


## Setting's 


In [30]:
from pathlib import Path
import csv, json, logging, os, sys, math
from importlib.metadata import distribution, version


In [2]:
from tracr.rasp import rasp
from tracr.compiler import compiling, lib, basis_inference, craft_graph_to_model, expr_to_craft_graph, nodes, rasp_to_graph
from tracr.craft import bases, transformers


In [3]:
VOCAB = {
    -2,
    -1,
    0,
    1,
    2,
    3,
    4,
    5,
}


In [4]:
MAX_SEQ_LEN = 6
CASUAL      = False
BOS, PAD    = "BOS", "PAD"
MLP_EXACTNESS = 100 


In [5]:
tracr_version = version("tracr")
direct_url    = distribution("tracr").read_text("direct_url.json")
tracr_commit  = (
    json.loads(direct_url).get("vcs_info", {}).get("commit_id")
    if direct_url else None
)

In [6]:
python_version = sys.version.split()[0]
python_version

'3.12.14'

In [7]:
project_root = Path.cwd()
print(project_root)

notebooks


In [8]:
logging_directory = project_root / "results" / "logs"
logging_directory.mkdir(parents=True, exist_ok=True)
logging_path = logging_directory / "tracr.log"

In [9]:
logger = logging.getLogger("tracr_step1")
logger.setLevel(logging.INFO)


In [10]:
for handler in logger.handlers:
    handler.close()
logger.handlers.clear()

In [11]:
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
for handler in (
    logging.FileHandler(logging_path, mode="w"),
    logging.StreamHandler(),
):
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info(
    "START tracr=%s commit=%s python=%s",
    tracr_version, tracr_commit, python_version,
)

2026-09-30 12:55:45,987 | INFO | START tracr=1.0.0 commit=9ce2b8c82b6ba10e62e86cf6f390e7536d4fd2cd python=3.12.14


## programs

#### Absolute Value

In [12]:
p01_absolute = rasp.Map(
    lambda x: abs(x),
    rasp.tokens,
).named("absolute")

#### Index parity

In [ ]:
p02_index_parity = rasp.SequenceMap(
    lambda token, index: index % 2,
    rasp.tokens,
    rasp.indices,
).named("index_parity")

#### Increment by index

In [14]:
p03_increment_by_index = rasp.SequenceMap(
    lambda token, index: token + index,
    rasp.tokens,
    rasp.indices,
).named("increment_by_index")

#### First Element

In [15]:
first_position = rasp.Select(
    rasp.indices,
    rasp.indices,
    lambda key, query: key == 0,
).named("first_position")

p04_first_element = rasp.Aggregate(
    first_position,
    rasp.tokens,
).named("first_element")

#### Histogram

In [17]:
p05_histogram = lib.make_hist().named("histogram")

#### Count greater than

In [18]:
greater_than_selector = rasp.Select(
    rasp.tokens,
    rasp.tokens,
    rasp.Comparison.GT,
).named("greater_than_selector")

p06_count_greater_than = rasp.SelectorWidth(
    greater_than_selector
).named("count_greater_than")

#### Sum with next
The final value is added to itself

In [20]:
length_for_next = lib.make_length()

next_clamped_index = rasp.SequenceMap(
    lambda index, length: min(index + 1, length - 1),
    rasp.indices,
    length_for_next,
).named("next_clamped_index")

next_selector = rasp.Select(
    rasp.indices,
    next_clamped_index,
    rasp.Comparison.EQ,
).named("next_selector")

next_value = rasp.Aggregate(
    next_selector,
    rasp.tokens,
).named("next_value")

p07_sum_with_next = rasp.SequenceMap(
    lambda current, following: current + following,
    rasp.tokens,
    next_value,
).named("sum_with_next")

#### Pairwise sum
The first value pairs with the final value

In [22]:
length_for_previous = lib.make_length()

previous_cyclic_index = rasp.SequenceMap(
    lambda index, length: (index - 1) % length if length else 0,
    rasp.indices,
    length_for_previous
).named("previous_cyclic_index")

previous_selector = rasp.Select(
    rasp.indices,
    previous_cyclic_index,
    rasp.Comparison.EQ,
).named("previous_selector")

previous_value = rasp.Aggregate(
    previous_selector,
    rasp.tokens,
).named("previous_value")

p08_pairwise_sum = rasp.SequenceMap(
    lambda current, previous: current + previous,
    rasp.tokens,
    previous_value,
).named('pairwise_sum')

#### Check Increasing

Equal adjacent values are allowed in this program

In [23]:
previous_for_order = lib.shift_by(1, rasp.tokens)

locally_increasing = rasp.SequenceMap(
    lambda current, previous: (
        True if previous is None else previous <= current
    ),
    rasp.tokens,
    previous_for_order,
).named("locally_increasing")

decrease_selector = rasp.Select(
    locally_increasing,
    rasp.indices,
    lambda is_ok, query: is_ok is False,
).named("decrease_selector")

decrease_count = rasp.SelectorWidth(
    decrease_selector
).named("decrease_count")

p09_check_increasing = rasp.Map(
    lambda count: 1 if count == 0 else 0,
    decrease_count,
).named("check_increasing")

#### Rotate left

In [24]:
sequence_length = lib.make_length()

next_index = rasp.SequenceMap(
    lambda index, length: 0 if length == 0 else (index + 1) % length,
    rasp.indices,
    sequence_length,
).named("next_index")

rotate_selector = rasp.Select(
    rasp.indices,
    next_index,
    rasp.Comparison.EQ,
).named("rotate_selector")

p10_rotate_left = rasp.Aggregate(
    rotate_selector,
    rasp.tokens,
).named("rotate_left")

#### Check Palindrome

In [25]:
reversed_values = lib.make_reverse(rasp.tokens)

mirror_matches = rasp.SequenceMap(
    lambda current, mirror: current == mirror,
    rasp.tokens,
    reversed_values,
).named("mirror_matches")

mismatch_selector = rasp.Select(
    mirror_matches,
    rasp.indices,
    lambda matches, query: not matches,
).named("mismatch_selector")

mismatch_count = rasp.SelectorWidth(
    mismatch_selector
).named("mismatch_count")

p11_check_palindrome = rasp.Map(
    lambda count: 1 if count == 0 else 0,
    mismatch_count,
).named("check_palindrome")

#### Sorting

In [26]:
p12_sorting = lib.make_sort(
    rasp.tokens,
    rasp.tokens,
    max_seq_len=MAX_SEQ_LEN,
    min_key=1.0,
).named("sorting")

### Program Listing

In [27]:
PROGRAMS = {
    "P01_absolute": p01_absolute,
    "P02_index_parity": p02_index_parity,
    "P03_increment_by_index": p03_increment_by_index,
    "P04_first_element": p04_first_element,
    "P05_histogram": p05_histogram,
    "P06_count_greater_than": p06_count_greater_than,
    "P07_sum_with_next": p07_sum_with_next,
    "P08_pairwise_sum": p08_pairwise_sum,
    "P09_check_increasing": p09_check_increasing,
    "P10_rotate_left": p10_rotate_left,
    "P11_check_palindrome": p11_check_palindrome,
    "P12_sorting": p12_sorting,
}

## Logging

In [32]:
def extract_schedule(program, model):
    traced = rasp_to_graph.extract_rasp_graph(program)
    graph, sources, sink = traced.graph, traced.sources, traced.sink

    basis_inference.infer_bases(graph, sink, VOCAB, MAX_SEQ_LEN)
    expr_to_craft_graph.add_craft_components_to_rasp_graph(
        graph,
        bos_dir=bases.BasisDirection(rasp.tokens.label, BOS),
        mlp_exactness=MLP_EXACTNESS,
    )

    allocation = craft_graph_to_model._allocate_modules_to_layers(
        graph, sources
    )

    lane_index = {
        label: i for i, label in enumerate(model.residual_labels)
    }

    variables = []
    selectors = []

    for node_id, node in graph.nodes(data=True):
        expr = node[nodes.EXPR]

        if isinstance(expr, rasp.Select):
            selectors.append({
                "selector" : expr.label,
                "node_id"  : node_id,
            })

        if not isinstance(expr, rasp.SOp):
            continue

        block = node.get(nodes.MODEL_BLOCK)
        first = allocation.get(node_id)

        if isinstance(block, transformers.SeriesWithResiduals):
            first += 1 

        if first is None and expr.label not in (
            rasp.tokens.label,
            rasp.indices.label
        ):

            raise ValueError(f"No Writer allocated for {expr.label}")

        first_after = (
            "input embedding" if first is None else
            f"after {'attention' if first % 2 == 0 else 'mlp'}"
            f"{first // 2 + 1}"
        )

        labels = [
            str(direction)
            for direction in node[nodes.OUTPUT_BASIS]
        ]


        variables.append({
            "variable": expr.label,
            "residual_lane_indices": [
                lane_index[label] for label in labels
            ],
            "residual_lane_labels": labels,
            "first_write_half_layer_index": first,
            "first_write_after": first_after,
            "compiled_block": (
                "attention+mlp"
                if isinstance(block, transformers.SeriesWithResiduals)
                else type(block).__name__
                if block is not None else "input"
            ),
        })

    depth = craft_graph_to_model.compute_computational_depth(
        graph,
        [source[nodes.ID] for source in sources],
    )

    summary = {
        "transformer_layers": model.model_config.num_layers,
        "residual_width": len(model.residual_labels),
        "sequence_variable_count": len(variables),
        "computed_variable_count": sum(
            row["first_write_half_layer_index"] is not None
            for row in variables
        ),
        "selector_count": len(selectors),
        "max_dependency_depth": max(depth.values()),
    }

    return summary, variables, selectors
        

In [33]:
SAMPLES = [
    [1, 2, 1],
    [-2, 0, 3, -1],
    [5],
    [0, 0, 0, 0],
    [5, 4, 3, 2, 1, 0],
]

MODELS = {}
PROGRAM_SUMMARIES = []
VARIABLE_SCHEDULE = []
SELECTOR_SCHEDULE = []
evaluator = rasp.DefaultRASPEvaluator()

for name, program in PROGRAMS.items():
    logger.info("START %s", name)

    try:
        model = compiling.compile_rasp_to_model(
            program,
            vocab=VOCAB,
            max_seq_len=MAX_SEQ_LEN,
            causal=CASUAL,
            compiler_bos=BOS,
            compiler_pad=PAD,
            mlp_exactness=MLP_EXACTNESS,
        )
        summary, variables, selectors = extract_schedule(
            program, model
        )

        for tokens in SAMPLES:
            expected = list(evaluator.evaluate(program, tokens))
            actual = model.apply([BOS, *tokens]).decoded[1:]

            assert len(expected) == len(actual) and all(
                left == right or (
                    isinstance(left, (int, float))
                    and isinstance(right, (int, float))
                    and math.isclose(
                        left, right,
                        rel_tol=1e-6,
                        abs_tol=1e-6,
                    )
                )
                for left, right in zip(expected, actual)
            ), (
                f"{name}: input={tokens}, "
                f"RASP={expected}, model={actual}"
            )

        MODELS[name] = model
        PROGRAM_SUMMARIES.append({
            "program": name,
            **summary,
            "checks_passed": len(SAMPLES),
        })
        VARIABLE_SCHEDULE.extend(
            {"program": name, **row} for row in variables
        )
        SELECTOR_SCHEDULE.extend(
            {"program": name, **row} for row in selectors
        )

        logger.info(
            "PASS %s: %d layers, %d variables, %d/%d checks",
            name,
            summary["transformer_layers"],
            len(variables),
            len(SAMPLES),
            len(SAMPLES),
        )
        for row in variables:
            logger.info(
                "LANE %s: %s -> %s; %s",
                name,
                row["variable"],
                row["residual_lane_indices"],
                row["first_write_after"],
            )

    except Exception:
        logger.exception("FAILED %s", name)
        raise

print(
    f"Compiled {len(MODELS)}/{len(PROGRAMS)}; "
    f"validated {len(SAMPLES) * len(MODELS)} outputs"
)

2026-09-30 13:34:09,269 | INFO | START P01_absolute
2026-09-30 13:34:10,646 | INFO | PASS P01_absolute: 1 layers, 2 variables, 5/5 checks
2026-09-30 13:34:10,647 | INFO | LANE P01_absolute: absolute_1 -> [0, 1, 2, 3, 4, 5]; after mlp1
2026-09-30 13:34:10,647 | INFO | LANE P01_absolute: tokens -> [12, 13, 14, 15, 16, 17, 18, 19]; input embedding
2026-09-30 13:34:10,647 | INFO | START P02_index_parity
2026-09-30 13:34:11,706 | INFO | PASS P02_index_parity: 1 layers, 3 variables, 5/5 checks
2026-09-30 13:34:11,707 | INFO | LANE P02_index_parity: index_parity_2 -> [0, 1]; after mlp1
2026-09-30 13:34:11,707 | INFO | LANE P02_index_parity: tokens -> [9, 10, 11, 12, 13, 14, 15, 16]; input embedding
2026-09-30 13:34:11,707 | INFO | LANE P02_index_parity: indices -> [2, 3, 4, 5, 6, 7]; input embedding
2026-09-30 13:34:11,707 | INFO | START P03_increment_by_index
2026-09-30 13:34:12,598 | INFO | PASS P03_increment_by_index: 1 layers, 3 variables, 5/5 checks
2026-09-30 13:34:12,598 | INFO | LANE 

Compiled 12/12; validated 60 outputs


In [ ]:
from IPython.display import Markdown, display

results_dir = project_root / "results" / "tables"
results_dir.mkdir(parents=True, exist_ok=True)

settings = {
    "vocab": sorted(VOCAB),
    "max_seq_len": MAX_SEQ_LEN,
    "causal": CASUAL,
    "tracr_version": tracr_version,
    "tracr_commit": tracr_commit,
    "python_version": python_version,
}

schedule_path = results_dir / "step1_ground_truth_schedule.json"
schedule_path.write_text(json.dumps({
    "settings": settings,
    "programs": PROGRAM_SUMMARIES,
    "variables": VARIABLE_SCHEDULE,
    "selectors": SELECTOR_SCHEDULE,
}, indent=2) + "\n")


def lane_text(indices):
    if indices == list(range(indices[0], indices[-1] + 1)):
        return (
            str(indices[0]) if len(indices) == 1
            else f"{indices[0]}–{indices[-1]}"
        )
    return ", ".join(map(str, indices))


lines = [
    "# Tracr ground-truth schedules",
    "",
    (
        f"Tracr {tracr_version} (`{tracr_commit}`); "
        f"{len(MODELS)}/{len(PROGRAMS)} compiled; "
        f"{len(SAMPLES) * len(MODELS)} sampled outputs matched RASP."
    ),
    "",
]

for summary in PROGRAM_SUMMARIES:
    name = summary["program"]
    lines += [
        f"## {name}",
        "",
        "| Variable | Lanes | First appears |",
        "|---|---:|---|",
    ]

    rows = [
        row for row in VARIABLE_SCHEDULE
        if row["program"] == name
    ]
    rows.sort(key=lambda row: (
        -1 if row["first_write_half_layer_index"] is None
        else row["first_write_half_layer_index"],
        row["variable"],
    ))

    lines += [
        f"| {row['variable']} | "
        f"{lane_text(row['residual_lane_indices'])} | "
        f"{row['first_write_after']} |"
        for row in rows
    ]
    lines += ["", ""]

report = "\n".join(lines)
report_path = results_dir / "step1_ground_truth_schedule.md"
report_path.write_text(report + "\n")

logger.info("SAVED %s and %s", schedule_path, report_path)
display(Markdown(report))